In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import fmatoolbox as fma
import xarray as xr
import scipy as sp
import pathlib
froot = pathlib.Path().cwd().parent.parent / 'Results/Figures/ISIntervals'
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
do_save = False

In [ ]:
def _statesPSD(session,regs=None,when='sleep.*#0',bin=0.05,nperseg=8):

    R = fma.regions.regions(session,phases=when,states=['sws','rem'])
    regs = R.ids if regs is None else np.array(regs)[np.isin(regs,R.ids)]
    fr = R.firingRate(regs=regs,window=bin)

    states = ['sws','rem','other']
    psd = []
    for i, state in enumerate(states):
        fr_state = fma.general.restrict(fr,R.eventIntervals(state),s_ind=True)
        freq, p = sp.signal.welch(fr_state[:,1:],fs=1/bin,nperseg=2**nperseg,axis=0)
        psd.append(p)

    psd = xr.DataArray(psd,dims=['state','f','reg'],coords={'state': states, 'f': freq, 'reg': regs, 'rat': int(R.rat)})

    return psd


In [ ]:
session = fma.data.readBatchFile(batch_file)[0][16]
print(session)
psd = _statesPSD(session,bin=0.02,nperseg=2**11)

In [ ]:
fig, ax = fma.plotting.makeFigure(size=(35,5),n=(1,3))
for i, reg in enumerate(psd.reg.values):
    for j, state in enumerate(psd.state.values):
        ax[j].axvline(0.1,ls='--',color='gray')
        ax[i].semilogy(psd.f,psd.sel(reg=reg,state=state)) #,color=isru.paperColors(reg))
        #ax[i].set_xlim([0,1])
ax[0].set(xlabel='frequency (Hz)',ylabel='PSD (V²/Hz)');